# EurecomGPT — Phase 2: train a tiny GPT & measure parallelism

Run this in **Google Colab** with a **GPU** runtime (Runtime -> Change runtime type -> T4 GPU). It exercises Lecture 3: CPU vectorization (SIMD), CPU threads and GPU (CUDA). The model code ships working; implementing it yourself (from `templates/`) is optional. The mandatory work is running this notebook end to end.

Run the cells **in order** — later cells reuse variables the earlier ones define (`data`, `sweep`, `cpu_s`, `GCS_URL`, ...), and the final report cell collects all of them. If you restart the runtime, start again from section 0.

## 0. Setup — clone your repo and install dependencies

Colab starts with an empty disk, so first pull in your own repository: that is where your `model.py`, `attention_numpy.py` and the helper modules live. Two packages are not preinstalled in Colab — `safetensors` (the weight format we save in) and `google-cloud-storage` (used in section 6 to publish the model).

In [ ]:
# Edit the URL to YOUR repo — this is the code the rest of the notebook imports.
!git clone https://github.com/<you>/<your-repo>.git repo 2>/dev/null || true
%cd repo
!pip -q install safetensors google-cloud-storage
# Phase-2 modules are in a subfolder, so add it to the import path.
import sys; sys.path.insert(0, 'phase-2-tiny-gpt')

### Imports, and confirm the GPU is really attached

What each import is for:

- `attention_naive` / `attention_vectorized` — the two implementations of the *same* attention maths that section 1 times against each other.
- `CONFIG` — the model's hyperparameters (layers, heads, embedding width). Everything downstream builds the model from this one object.
- `GPT` — the transformer itself.
- `train` / `experiments` — provided helpers: batching, the training loop, sampling, the timing harnesses and the report writer.

**Check the printed device.** If it says `cpu`, the GPU runtime is not attached and section 4 will show no speedup — fix it with Runtime -> Change runtime type -> T4 GPU before going on.

In [ ]:
import numpy as np, torch
from attention_numpy import attention_naive, attention_vectorized
from config import CONFIG
from model import GPT
import train, experiments

# 'cuda' only if Colab actually gave us a GPU runtime; everything else falls back to CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

## 1. Attention: naive loops vs vectorized (SIMD)

Both functions compute *identical* attention output — the only difference is how they get there. `attention_naive` walks the sequence with Python loops, one dot product at a time. `attention_vectorized` hands whole matrices to NumPy, which dispatches to a BLAS library compiled with AVX/SIMD instructions that process several floats per CPU cycle.

`T=256` is the sequence length and `d=64` the head dimension — the same shapes the real model uses, so the timing is representative.

**What to look for:** a large speedup (often 100x or more) from code that is mathematically the same. `np.show_config()` prints which BLAS NumPy is linked against — that library is where the speedup actually comes from.

In [ ]:
# Same maths, timed two ways. T = sequence length, d = head dimension.
t_naive = experiments.time_attention(attention_naive, T=256, d=64)
t_vec   = experiments.time_attention(attention_vectorized, T=256, d=64)
print(f'naive      : {t_naive:.2f} ms')
print(f'vectorized : {t_vec:.2f} ms   ({t_naive / t_vec:.1f}x faster)')

# Which BLAS is NumPy using? This is what makes the vectorized version fast.
np.show_config()

## 2. Load the training corpus

Everything from here on needs training data, so load it once into `data` and reuse it.

The text is TinyShakespeare (~1 MB). If your repo already contains `input.txt` it is used directly; otherwise it is downloaded. `train.encode` turns the characters into integer token ids, and the result becomes one long 1-D tensor that later cells slice batches out of.

**What to look for:** a token count around one million. A much smaller number means the download failed and later training will not converge.

In [ ]:
import os, urllib.request

# Prefer a local copy if the repo has one; otherwise fetch TinyShakespeare (~1 MB).
if os.path.exists('phase-2-tiny-gpt/input.txt'):
    text = open('phase-2-tiny-gpt/input.txt', encoding='utf-8', errors='replace').read()
else:
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    text = urllib.request.urlopen(url).read().decode('utf-8', 'replace')

# Encode characters -> token ids, as one long 1-D tensor to slice batches from.
data = torch.tensor(train.encode(text), dtype=torch.long)
print('corpus tokens:', len(data))

## 3. Multi-threading: sweep CPU threads (intra-op parallelism)

PyTorch splits a single CPU operation — a matrix multiply, say — across `torch.set_num_threads(n)` worker threads. This is *intra-op* parallelism: one operation, many cores, as opposed to running several operations at once.

The cell below builds a fresh model, defines one training step (fetch a batch, forward, backward), and times that step at 1, 2, 4 and 8 threads.

**What to look for:** time per step falls as threads are added, then flattens or gets *worse*. That turning point is the contention point — the moment coordination costs more than the extra parallelism buys — and explaining where it lands is what your reflection is about.

In [ ]:
model = GPT(CONFIG)
print('parameters:', model.num_params())

# One training step: sample a batch, forward pass, backward pass. This is what gets timed.
def one_step():
    x, y = train.get_batch(data, CONFIG.block_size, 16, 'cpu')
    _, loss = model(x, y); loss.backward()

# Time that step at each thread count. Returns {'1': ms, '2': ms, ...}.
sweep = experiments.threading_sweep(one_step, threads=(1, 2, 4, 8))
print('threads -> ms:', sweep)

### Plot the sweep

The same numbers as a curve, because the shape is the point — you are looking for where it stops going down. **Save this plot**: the deliverables ask for it, and your reflection should refer to it.

In [ ]:
import matplotlib.pyplot as plt

# sweep keys are strings ('1','2',...); sort numerically so the x-axis is in order.
ts = sorted(int(k) for k in sweep)
plt.plot(ts, [sweep[str(t)] for t in ts], 'o-'); plt.xlabel('threads'); plt.ylabel('ms/step')
plt.title('CPU intra-op threading'); plt.grid(True); plt.show()

## 4. GPU (CUDA): the same training loop on a T4

Identical model, identical data, identical number of steps — the only thing that changes is the device. A GPU runs thousands of arithmetic units in parallel, which suits the dense matrix multiplies that dominate a transformer.

Two fresh models are built so neither run inherits the other's trained weights. `log_every=0` silences per-step logging so the timing is not distorted by printing.

**What to look for:** a large ratio — often 50x or more. If it is close to 1x, you are on a CPU runtime (check the device printed in section 0).

In [ ]:
# Same 100 steps on each device; two fresh models so neither reuses the other's weights.
cpu_loss, cpu_s = train.train(GPT(CONFIG), data, steps=100, device='cpu', log_every=0)
gpu_model = GPT(CONFIG)
gpu_loss, gpu_s = train.train(gpu_model, data, steps=100, device=device, log_every=0)
print(f'CPU 100 steps: {cpu_s*1000:.0f} ms  |  {device} 100 steps: {gpu_s*1000:.0f} ms',
      f'  ({cpu_s/gpu_s:.1f}x)')

## 5. Train the model, generate a sample, save the weights

The real training run: 2000 steps on whichever device section 0 detected. This is the model you will publish and that Phase 5 will later serve.

`train.sample` then generates 200 tokens starting from a newline, and `save_model` writes `model.safetensors`.

**What to look for:** the loss should fall well below the random-initialisation value of about 5.55 — under 3.0 is the bar. The sample will not be Shakespeare, but it should look like text: words of plausible length, line breaks, capital letters after full stops. If the loss is still near 5.5 or the sample is noise, something upstream is wrong; do not publish it.

In [ ]:
model = GPT(CONFIG)
final_loss, secs = train.train(model, data, steps=2000, device=device, log_every=200)
print(f'final loss {final_loss:.3f} in {secs:.1f}s')

# Generate from a newline so the model starts at the beginning of a line.
sample_text = train.sample(model, prompt='\n', max_new_tokens=200, device=device)
print('----- sample -----\n' + sample_text)

# Write the weights to disk; the next section uploads this file.
train.save_model(model, 'model.safetensors')

## 6. Upload the weights to Cloud Storage (public)

The grader cannot see your Colab session, so the model has to be published somewhere reachable — a public Cloud Storage object. Phase 5 will load the same URL to serve your model.

Colab is **not** logged into `gcloud`, so authenticate this session first. The popup asks for the Google account that owns your Phase-0 project.

In [ ]:
from google.colab import auth
auth.authenticate_user()   # opens a popup to log into your Google account

Now create the bucket (if it does not exist), make it publicly readable, and upload.

**Set `PROJECT` to your own Phase-0 project id before running this.** Bucket names are globally unique across all of Google Cloud, which is why the project id is used as a prefix. The model is a few MB, comfortably inside the 5 GB free tier.

**What to look for:** the printed URL should download the file when you open it in a browser. Copy it — the report needs it, and so does Phase 5.

In [ ]:
PROJECT = 'REPLACE-with-your-project-id'   # <-- your Phase-0 GCP project id
BUCKET  = f'{PROJECT}-eurecomgpt'           # bucket names are globally unique

from google.cloud import storage
client = storage.Client(project=PROJECT)

# Reuse the bucket if a previous run (or Phase 3) already created it.
try:
    bucket = client.get_bucket(BUCKET)
except Exception:
    bucket = client.create_bucket(BUCKET, location='US')

# Grant allUsers read access, so the grader can fetch the model without credentials.
# Bucket-level IAM is used because it works with uniform bucket-level access.
policy = bucket.get_iam_policy(requested_policy_version=3)
policy.bindings.append({'role': 'roles/storage.objectViewer', 'members': {'allUsers'}})
bucket.set_iam_policy(policy)

blob = bucket.blob('model.safetensors')
blob.upload_from_filename('model.safetensors')
GCS_URL = f'https://storage.googleapis.com/{BUCKET}/model.safetensors'
print('public URL:', GCS_URL)

## 7. Write the submission report

This collects every measurement the notebook has taken into `submission/phase2_report.json` — the attention timings from section 1, the thread sweep from section 3, the CPU/GPU comparison from section 4, the training result and sample from section 5, and the public model URL from section 6.

That is why the cells must be run in order: this one fails with a `NameError` if any earlier section was skipped, and the name it complains about tells you which.

In [ ]:
experiments.write_report(
    attention={'naive_ms': t_naive, 'vectorized_ms': t_vec, 'speedup': round(t_naive/t_vec, 2)},
    threads=sweep,
    devices={'cpu_ms': round(cpu_s*1000, 1), 'gpu_ms': round(gpu_s*1000, 1)},
    training={'steps': 2000, 'final_loss': round(final_loss, 4), 'param_count': model.num_params()},
    sample_text=sample_text,
    gcs_url=GCS_URL,
)

Now **commit** `submission/phase2_report.json` and push (see TASKS.md). Your CI + the instructor grade the report + your public model.